# Pseudo-labeling & entropy minimization

_Semi & Self-Supervised Learning_

**Let the model label its own unlabeled data, keep only the predictions it is sure about, and train on those.**

You have a few labeled examples and a mountain of unlabeled ones. Pseudo-labeling (also called self-training) is the simplest way to use that mountain.

       The recipe is a loop: (1) train on the labels you have, (2) run the model on the unlabeled data, (3) for the examples where the model is very confident, pretend its guess is the true label, (4) add those "pseudo-labeled" examples to the training set and retrain.

---

This notebook is a practice scaffold for the **Pseudo-labeling & entropy minimization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def pseudo_label_step(model, x_labeled, y_labeled, x_unlabeled,
                      optimizer, tau=0.95, lambda_u=1.0):
    """One semi-supervised step: supervised loss + confidence-gated pseudo-label loss."""
    model.train()

    # ---- supervised term on the real labels ----
    logits_l = model(x_labeled)                       # (B_l, C)
    loss_sup = F.cross_entropy(logits_l, y_labeled)

    # ---- pseudo-label term on the unlabeled batch ----
    with torch.no_grad():                             # don't backprop through target creation
        logits_u = model(x_unlabeled)                 # (B_u, C)
        probs_u = F.softmax(logits_u, dim=1)          # predicted class probabilities p
        max_p, y_hat = probs_u.max(dim=1)             # top prob max_c p_c and argmax y_hat
        mask = (max_p > tau).float()                  # 1[max_c p_c > tau] : keep confident ones

    # forward AGAIN with grad so the pseudo-label loss can train the model
    logits_u_grad = model(x_unlabeled)                # (B_u, C)
    # per-example cross-entropy to the HARD pseudo-label y_hat, then gate by the mask
    ce_u = F.cross_entropy(logits_u_grad, y_hat, reduction="none")  # -log p_{y_hat}
    loss_u = (mask * ce_u).sum() / (mask.sum() + 1e-8)             # average over kept examples

    loss = loss_sup + lambda_u * loss_u               # total loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    kept = int(mask.sum().item())                     # how many unlabeled examples were trusted
    return loss.item(), loss_sup.item(), loss_u.item(), kept

# --- optional: the soft entropy-minimization alternative to the hard gate ---
def entropy_min_loss(logits_unlabeled):
    """-sum_c p_c log p_c, averaged: pushes every prediction to be confident (low entropy)."""
    p = F.softmax(logits_unlabeled, dim=1)
    log_p = F.log_softmax(logits_unlabeled, dim=1)    # numerically stable log p
    return -(p * log_p).sum(dim=1).mean()             # mean entropy over the batch

## Visualize the data & results

_On real digit images with only 40 labels, how does self-training accuracy depend on the confidence threshold — where is the sweet spot, and where does the wrong threshold either admit bad pseudo-labels (confirmation bias) or admit nothing (no learning)?_

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X, y = digits.data / 16.0, digits.target     # pixels scaled to 0..1
thresholds = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
make = lambda: LogisticRegression(max_iter=200)   # calibrated-ish softmax base

acc = np.zeros(len(thresholds)); base = []
for seed in range(8):                        # average over 8 stratified splits
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y)
    rng = np.random.RandomState(seed)
    y_mixed = np.full(ytr.shape, -1)         # -1 marks "unlabeled" for sklearn
    for c in range(10):                      # keep only 4 real labels per class = 40
        ci = np.where(ytr == c)[0]; rng.shuffle(ci); y_mixed[ci[:4]] = c

    m = y_mixed != -1                        # labels-only baseline
    base.append(make().fit(Xtr[m], y_mixed[m]).score(Xte, yte))

    for j, tau in enumerate(thresholds):     # self-train at each confidence threshold
        st = SelfTrainingClassifier(make(), threshold=tau, max_iter=10)
        st.fit(Xtr, y_mixed)                 # iteratively pseudo-labels the unlabeled pool
        acc[j] += st.score(Xte, yte)
acc /= 8

print("baseline:", round(float(np.mean(base)), 3))
for tau, a in zip(thresholds, acc):          # the exact plotted numbers
    print(f"tau={tau:.2f} acc={a:.3f}")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Two classes, threshold $\tau = 0.9$. The model predicts $p = (0.93, 0.07)$ on an unlabeled image. Does it get a pseudo-label, what is it, and what is the loss term?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the top probability $\max_c p_c$ and compare it to $\tau$. — _The gate $\mathbb{1}[\max_c p_c > \tau]$ decides whether this example is used at all._
- If the gate is open, the pseudo-label is $\hat{y} = \arg\max_c p_c$ — the index of the biggest probability. — _Hard pseudo-labeling commits to the single most likely class._
- Compute the cross-entropy term $-\log p_{\hat{y}}$. — _That is the loss the example contributes, pushing $p_{\hat{y}}$ toward 1._

**Answer:** $\max_c p_c = 0.93 \gt  0.9$, so the gate opens. The pseudo-label is $\hat{y} = 1$ (class 1, the 0.93). The loss term is $-\log(0.93) = 0.073$. It is small and positive, nudging the model to be even surer of class 1 on this image.

</details>

**Problem 2.** Explain, in plain words, why pushing predictions on unlabeled data to be confident (low entropy) tends to move the decision boundary into a low-density region — and when this backfires.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what high vs low entropy means about where a point sits. — _High-entropy points are near the boundary (model is torn); low-entropy points are deep inside a cluster._
- Ask what minimizing entropy over all unlabeled points does to the boundary. — _Forcing confidence on every point repels the boundary from where points actually are._
- Consider the case where two classes genuinely overlap. — _If the densest region is the true boundary, demanding confidence there is forcing wrong, overconfident calls._

**Answer:** A point near the boundary gets a torn, high-entropy prediction; a point inside a cluster gets a confident, low-entropy one. Minimizing entropy demands confidence everywhere, which pushes the boundary out of the dense clusters and into the empty gaps between them — the low-density-separation idea. It backfires when classes truly overlap: then the densest area is the boundary, and forcing confident labels there bakes in errors. Those confident mistakes become training targets, which is the start of confirmation bias.

</details>